In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import UEG_response as ur
import json

# import matplot2tikz 

from UEG_response import _Maldague_chi_2_0_CV, _reduced_FD

In [ ]:
# Conditions
rs = 3.23
theta = 2.0

# Units
hbar = 1.0
aB = 1.0
m = 1.0
e = 1.0

# Normalisation
qF = (9*np.pi/4)**(1/3) / (rs*aB)
EF = hbar**2 * qF**2 / (2*m)
beta = 1/(theta*EF)
n = 3/(4*np.pi*rs**3)

# Tolerances
reltol = 1e-16
abstol = 1e-8
eta_log  = 1e-6
eta_sqrt = 1e-6
eta_pol = 1e-4
points_n = 5
tol_upper = 1e-8
dx = 1e-4
lower = 1e-6
limit = 50
ms = 2


In [ ]:
# Plot direct integrand


k1 = 4.85 * qF
omega1 = -0.3 * EF / hbar
k2 = 2.29 * qF
omega2 = 0.03 * EF / hbar
csTheta = 0.9

sng1 = 1
sng2 = -1

z1 = hbar * omega1 / EF
y1 = k1 / qF
z2 = hbar * omega2 / EF
y2 = k2 / qF


A  = (-z1 - y1**2)/(2*y1)
B  = (-z2 - y2**2)/(2*y2)
G2 = A**2 - 2*A*B*csTheta + B**2
snTheta2 = 1 - csTheta**2

eta_log = 0.1
plt.fill_between([np.abs(A)-eta_log, np.abs(A)+eta_log], 10, -15, facecolor='green', alpha=.5, label="Analytical treatment")
plt.fill_between([np.abs(B)-eta_log, np.abs(B)+eta_log], 10, -15, facecolor='green', alpha=.5)

eta_sqrt = 0.1
xG = np.sqrt(G2/snTheta2)
plt.fill_between([xG-eta_sqrt, xG+eta_sqrt], 10, -15, facecolor='green', alpha=.5)


xLow  = min(np.abs(A), np.abs(B))
xHigh = max(np.abs(A), np.abs(B))
x = np.concatenate((np.linspace(0.0, xLow-eta_log, 30),
                    np.linspace(xLow-eta_log, xLow+eta_log, 200),
                    np.linspace(xLow+eta_log, xHigh-eta_log, 30),
                    np.linspace(xHigh-eta_log, xHigh+eta_log, 500),
                    np.linspace(xHigh+eta_log, xG-eta_sqrt, 30),
                    np.linspace(xG-eta_sqrt, xG+eta_sqrt, 100),
                    np.linspace(xG+eta_sqrt, 5.0, 50),
                    ))

# Evaluation from the paper
inv_theta = 1/theta
eta = beta * ur.compute_chemical_potential(n, hbar, m, beta, reltol=reltol, ms=ms)
val = _reduced_FD(x, inv_theta, eta) * ur.phi_2_corrected(x, A, sng1, B, sng2, csTheta)
plt.plot(x, np.real(val), '-k',  label='Real part')
plt.plot(x, np.imag(val), '--r', label='Imag part')


plt.legend()

plt.xlabel(r'$x$')
plt.ylabel(r'$f(q_F x)\, \varphi(x; A, B, \cos\theta)$')


plt.ylim([-0.3, 0.5])
plt.xlim([np.min(x), np.max(x)])

# plt.savefig("figures/integrand_direct.jpg", dpi=400, bbox_inches='tight')
# matplot2tikz.save("figures/integrand_direct.tex")


In [ ]:
# Plot the Maldague integrand for quadratic response

eta_bar = np.linspace(lower, 8, 1000)

# Compuet chemical potential.
eta = beta * ur.compute_chemical_potential(n, hbar, m, beta, reltol=reltol, ms=ms)

tmp, points_I = _Maldague_chi_2_0_CV(eta_bar, k1, omega1, k2, omega2, csTheta, beta, hbar, m, ms)
res = tmp / (4 * np.cosh((eta_bar - eta)/2)**2)

# Set interval
max_suppression = 1e-30
eta_or_zero = max(0.0, eta)
low  = max(lower, eta_or_zero + np.log(tol_upper))
high = min( max(eta_or_zero - np.log(tol_upper), np.max(points_I[:, 0])+3, np.max(points_I[:, 1])+3), eta - np.log(max_suppression)) # Make sure non-zero part of imag is included but also limit max suppression.


plt.plot(eta_bar, np.abs(np.real(res))/np.max(np.abs(np.real(res))), '-k', label="Real part")
plt.plot(eta_bar, np.abs(np.imag(res))/np.max(np.abs(np.imag(res))), '--r', label="Imag part")

for p in points_I.flatten():
    if (p > low and p < high):
        plt.axvline(p, linestyle=':', color='b', alpha=0.3)

# points_FD = np.array( [(max(0.0, eta) + n) for n in range(-points_n, points_n+1)] ) # Special points from thermal factor.
# for p in points_FD:
#     if (p > low and p < high):
#         plt.axvline(p, linestyle=':', color='g')

# plt.axvline(low, linestyle='-', color='y')
# plt.axvline(high, linestyle='-', color='y')

plt.legend(loc='upper right')

plt.yscale('log')
# plt.xscale('log')

plt.ylabel(r"Norm. Maldague integrand")
plt.xlabel(r"$\bar{\eta}$")

# plt.xlim(right=np.max(eta_bar))

plt.ylim(bottom=1e-3)
plt.ylim(top=2.0)
plt.xlim(left=0.0)
plt.xlim(right=np.max(eta_bar))

# plt.savefig("figures/integrand_Maldague.jpg", dpi=400, bbox_inches='tight')
# matplot2tikz.save("figures/integrand_Maldague.tex")


In [ ]:
eta